***Json Data***

In [0]:
json_data = {
    "entities": [
        {
            "id": 101,
            "name": "Amit",
            "data": {
                "attributes": [
                    {
                        "name": "city",
                        "properties": {
                            "value": "Gwalior"
                        }
                    }
                ],
                "address": {
                    "city": "Gwalior",
                    "state": "MP"
                }
            },
            "relationships": [
                {
                    "type": "friend",
                    "target": 202
                }
            ]
        }
    ]
}

df = spark.createDataFrame([json_data])

df.printSchema()

***Create a DataFrame***

In [0]:
import json
from pyspark.sql.types import StructType

# Extract entities from json_data
entities = json_data["entities"]

# Convert each entity to JSON string for schema inference
json_strings = [json.dumps(entity) for entity in entities]

# Create a simple DataFrame with JSON strings, then parse
temp_data = [(s,) for s in json_strings]
temp_df = spark.createDataFrame(temp_data, ["json_str"])

# Use from_json to parse with inferred schema
from pyspark.sql.functions import from_json, schema_of_json

sample_schema = schema_of_json(json_strings[0])
df = temp_df.select(from_json("json_str", sample_schema).alias("parsed")).select("parsed.*")

In [0]:
# Print schema in tree format
df.printSchema()

# Display data
display(df)

In [0]:
schema=df.schema


In [0]:
flatten_attributs=["data"]

Explode_attributes=["data.attributes"]

In [0]:
from pyspark.sql import functions as F
for attr in flatten_attributs:
    fields=schema[attr].dataType.fieldNames()

    for field in fields:
        display(f"{attr}_{field}",
                F.col(f"{attr}.{field}"))
        
    df=df.drop(attr)
        

In [0]:
for attr in Explode_attributes:
    df = df.withColumn(attr.replace(".", "_"), F.explode_outer(F.col(attr)))